# Enhanced HCI Application: Face-Controlled Game Interface

This notebook demonstrates the enhanced version of the face-controlled game interface with improved usability and accessibility features.

## Key Improvements
1. **Adaptive Control Box**: The control area now scales based on screen resolution
2. **Visual Feedback**: On-screen indicators for actions and game states
3. **Calibration Phase**: Guided setup process for users
4. **Pause Function**: Added ability to pause/resume with SPACE key
5. **Performance Monitoring**: Real-time FPS counter
6. **Screen-Aware Positioning**: Game initialization adapts to screen size

## Controls
- **SPACE**: Start game (when face is centered) / Pause / Resume
- **ESC**: Exit application

Below is an example of what we can achieve:

In [57]:
from IPython.display import Video
Video('Output/Output_Pac_man.mov', width=800)

# Enhanced Face-Controlled Game Interface: Technical Deep Dive

This notebook provides a detailed explanation of implementing a face detection-based game controller using OpenCV and PyAutoGUI. The system uses computer vision to track face position and converts these movements into keyboard commands for game control.

## System Architecture

The interface consists of several key components:

1. **Face Detection Engine**:
   - Uses OpenCV's DNN module with a pre-trained SSD (Single Shot Detector) model
   - Processes video frames in real-time
   - Returns face locations with confidence scores

2. **Control System**:
   - Adaptive control box that scales with screen size
   - State machine for movement tracking
   - Input conversion to keyboard commands

3. **User Interface**:
   - Visual feedback system
   - Calibration guidance
   - Performance metrics (FPS)

4. **State Management**:
   - Calibration phase
   - Running state
   - Pause functionality

## Implementation Overview

Below, we'll break down each component and explain its implementation in detail.

In [58]:
import cv2
import numpy as np
import pyautogui as gui
import time

# Set keypress delay to 0.
gui.PAUSE = 0

# Loading the pre-trained face model.
model_path = './model/res10_300x300_ssd_iter_140000.caffemodel'
prototxt_path = './model/deploy.prototxt'

# Control states
GAME_STATES = {
    'CALIBRATION': 0,
    'RUNNING': 1,
    'PAUSED': 2
}

## Face Detection and Visual Feedback
These functions handle face detection and provide visual feedback to the user:

In [59]:
def detect(net, frame):
    '''
    Detect faces in the frame using the pre-trained model.
    Returns a list of detected faces with positions and confidence scores.
    '''
    detected_faces = []
    (h, w) = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(
        cv2.resize(frame, (300, 300)),
        1.0,
        (300, 300),
        (104.0, 177.0, 123.0))
    net.setInput(blob)
    detections = net.forward()
    for i in range(0, detections.shape[2]):
        confidence = detections[0, 0, i, 2]
        if confidence > 0.5:
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            (startX, startY, endX, endY) = box.astype("int")
            detected_faces.append({
                'start': (startX, startY),
                'end': (endX, endY),
                'confidence': confidence})
    return detected_faces

def drawFace(frame, detected_faces, action=None):
    '''
    Draw boxes around detected faces and show current action.
    '''
    for face in detected_faces:
        # Draw face rectangle in green
        cv2.rectangle(frame, face['start'], face['end'], (0, 255, 0), 3)
        
        # Show action feedback if provided
        if action:
            text_pos = (face['start'][0], face['start'][1] - 10)
            cv2.putText(frame, action, text_pos, 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    return frame

## Adaptive Control Box
The control box now adjusts based on screen dimensions:

In [60]:
def calculate_control_box(frame_width, frame_height):
    '''
    Calculate an adaptive control box size based on frame dimensions.
    '''
    box_width = min(300, frame_width // 3)
    box_height = min(400, frame_height // 2)
    
    left_x = frame_width // 2 - box_width // 2
    right_x = frame_width // 2 + box_width // 2
    top_y = frame_height // 2 - box_height // 2
    bottom_y = frame_height // 2 + box_height // 2
    
    return [left_x, right_x, bottom_y, top_y]

## Movement Detection and Control
Enhanced movement detection with visual feedback:

In [61]:
def checkRect(detected_faces, bbox):
    '''
    Check if face is within the control box.
    '''
    for face in detected_faces:
        x1, y1 = face['start']
        x2, y2 = face['end']
        if x1 >= bbox[0] and x2 <= bbox[1] and y1 >= bbox[3] and y2 <= bbox[2]:
            return True
    return False

def move(detected_faces, bbox):
    '''
    Convert face position to keyboard commands and provide visual feedback.
    '''
    global last_mov
    action = None
    
    for face in detected_faces:
        x1, y1 = face['start']
        x2, y2 = face['end']

        # Center
        if checkRect(detected_faces, bbox):
            last_mov = 'center'
            action = 'CENTER'
            return action

        elif last_mov == 'center':
            # Direction controls with visual feedback
            if x1 < bbox[0]:
                gui.press('left')
                last_mov = 'left'
                action = 'LEFT'
            elif x2 > bbox[1]:
                gui.press('right')
                last_mov = 'right'
                action = 'RIGHT'
            if y2 > bbox[2]:
                gui.press('down')
                last_mov = 'down'
                action = 'DOWN'
            elif y1 < bbox[3]:
                gui.press('up')
                last_mov = 'up'
                action = 'UP'

    return action

## User Interface
Functions for managing the UI and calibration:

In [62]:
def show_calibration_guide(frame):
    '''
    Display calibration instructions with centered text.
    '''
    h, w = frame.shape[:2]
    text = "Position your face in the box and press SPACE to start"
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.6
    thickness = 1
    
    # Get text size
    (text_width, text_height), baseline = cv2.getTextSize(text, font, font_scale, thickness)
    
    # Calculate text position to center it
    text_x = (w - text_width) // 2
    text_y = h // 2
    
    cv2.putText(frame, text, (text_x, text_y), font, font_scale, (0, 255, 0), thickness)
    return frame

def play(prototxt_path, model_path):
    '''
    Main game loop with enhanced features.
    '''
    # Initialize camera
    cap = cv2.VideoCapture(0)
    
    # Check if camera opened successfully
    if not cap.isOpened():
        print(
            "Error: Could not open camera")
        return
    
    # Getting the Frame width and height
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Co-ordinates of the bounding box on frame
    bbox = calculate_control_box(frame_width, frame_height)
    
    net = cv2.dnn.readNetFromCaffe(prototxt_path, model_path)
    
    state = GAME_STATES['CALIBRATION']
    last_time = time.time()
    
    while True:
        ret, frame = cap.read()
        if not ret:
            print(
                "Error: Can't receive frame from camera")
            break
            
        frame = cv2.flip(frame, 1)
        detected_faces = detect(net, frame)
        
        if state == GAME_STATES['CALIBRATION']:
            frame = show_calibration_guide(frame)
            if checkRect(detected_faces, bbox):
                if cv2.waitKey(1) & 0xFF == ord(' '):
                    state = GAME_STATES['RUNNING']
                    
        elif state == GAME_STATES['RUNNING']:
            action = move(detected_faces, bbox)
            frame = drawFace(frame, detected_faces, action)
            if cv2.waitKey(1) & 0xFF == ord(' '):
                state = GAME_STATES['PAUSED']
                
        elif state == GAME_STATES['PAUSED']:
            if cv2.waitKey(1) & 0xFF == ord(' '):
                state = GAME_STATES['RUNNING']
                
        # Drawing the control rectangle in the center of the frame
        frame = cv2.rectangle(
            frame, (bbox[0], bbox[3]), (bbox[1], bbox[2]), (0, 0, 255), 5)
        
        # Show FPS
        current_time = time.time()
        fps = 1 / (current_time - last_time)
        last_time = current_time
        cv2.putText(frame, f'FPS: {fps:.2f}', (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        # Show state
        cv2.putText(frame, f'State: {list(GAME_STATES.keys())[state]}', (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        # Display the frame
        cv2.imshow('Face-Controlled Game Interface', frame)
        
        # Check for ESC key (key code 27)
        key = cv2.waitKey(1) & 0xFF
        if key == 27:  # ESC key
            print(
                "Exiting...")
            break
    
    # Properly release resources
    cap.release()
    cv2.destroyAllWindows()

## 4. Main Game Loop

The main loop integrates all components and manages the game state:

1. **Initialization**:
   - Load face detection model
   - Set up video capture
   - Calculate control box dimensions

2. **State Management**:
   - Handle transitions between states
   - Process user input (SPACE/ESC)
   - Manage game controls

3. **Frame Processing**:
   - Capture and process video frames
   - Update visual feedback
   - Calculate performance metrics

In [63]:
def play(prototxt_path, model_path):
    '''
    Main game loop with enhanced features.
    '''
    # Initialize camera
    cap = cv2.VideoCapture(0)
    
    # Check if camera opened successfully
    if not cap.isOpened():
        print(
            "Error: Could not open camera")
        return
    
    # Getting the Frame width and height
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Co-ordinates of the bounding box on frame
    bbox = calculate_control_box(frame_width, frame_height)
    
    net = cv2.dnn.readNetFromCaffe(prototxt_path, model_path)
    
    state = GAME_STATES['CALIBRATION']
    last_time = time.time()
    
    while True:
        ret, frame = cap.read()
        if not ret:
            print(
                "Error: Can't receive frame from camera")
            break
            
        frame = cv2.flip(frame, 1)
        detected_faces = detect(net, frame)
        
        if state == GAME_STATES['CALIBRATION']:
            frame = show_calibration_guide(frame)
            if checkRect(detected_faces, bbox):
                if cv2.waitKey(1) & 0xFF == ord(' '):
                    state = GAME_STATES['RUNNING']
                    
        elif state == GAME_STATES['RUNNING']:
            action = move(detected_faces, bbox)
            frame = drawFace(frame, detected_faces, action)
            if cv2.waitKey(1) & 0xFF == ord(' '):
                state = GAME_STATES['PAUSED']
                
        elif state == GAME_STATES['PAUSED']:
            if cv2.waitKey(1) & 0xFF == ord(' '):
                state = GAME_STATES['RUNNING']
                
        # Drawing the control rectangle in the center of the frame
        frame = cv2.rectangle(
            frame, (bbox[0], bbox[3]), (bbox[1], bbox[2]), (0, 0, 255), 5)
        
        # Show FPS
        current_time = time.time()
        fps = 1 / (current_time - last_time)
        last_time = current_time
        cv2.putText(frame, f'FPS: {fps:.2f}', (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        # Show state
        cv2.putText(frame, f'State: {list(GAME_STATES.keys())[state]}', (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        # Display the frame
        cv2.imshow('Face-Controlled Game Interface', frame)
        
        # Check for ESC key (key code 27)
        key = cv2.waitKey(1) & 0xFF
        if key == 27:  # ESC key
            print(
                "Exiting...")
            break
    
    # Properly release resources
    cap.release()
    cv2.destroyAllWindows()

## Running the Application
Initialize and start the face-controlled interface:

In [64]:
if __name__ == "__main__":
    last_mov = ''
    movements = []  # List to store movement history
    
    def log_movement(action):
        if action and action != 'CENTER':
            print(f"Movement detected: {action}")
            movements.append(action)
    
    # Modify the move function to log movements
    original_move = move
    def move_with_logging(detected_faces, bbox):
        action = original_move(detected_faces, bbox)
        log_movement(action)
        return action
    
    # Replace original move function with logging version
    move = move_with_logging
    
    try:
        play(prototxt_path, model_path)
    finally:
        # Print movement summary
        if movements:
            print("\nMovement Summary:")
            print(f"Total movements: {len(movements)}")
            for direction in ['UP', 'DOWN', 'LEFT', 'RIGHT']:
                count = movements.count(direction)
                print(f"{direction}: {count} times")

Movement detected: LEFT
Movement detected: RIGHT
Movement detected: UP
Movement detected: DOWN
Movement detected: UP
Movement detected: DOWN
Movement detected: DOWN
Movement detected: DOWN
Exiting...

Movement Summary:
Total movements: 8
UP: 2 times
DOWN: 4 times
LEFT: 1 times
RIGHT: 1 times


## Technical Considerations

### Performance Optimization
1. **Frame Processing**:
   - Efficient face detection with confidence thresholding
   - Optimized drawing operations
   - State-based processing

2. **Input Handling**:
   - Drift prevention through state tracking
   - Responsive keyboard control
   - Screen-aware positioning

### Best Practices
1. **Calibration**:
   - Center face in box
   - Maintain consistent lighting
   - Use deliberate movements

2. **Usage**:
   - Monitor FPS for performance
   - Use pause when needed
   - Keep face visible to camera

### Game Compatibility
The system works with any game that uses:
- Arrow key controls
- Simple directional inputs
- Pause functionality

## Example Applications

Here are some games tested with the interface: